In [10]:
import json
import os
import random
import re
from underthesea import word_tokenize

# Từ điển thay thế an toàn (Chỉ thay hư từ và danh từ chung)
SAFE_SYNONYMS = {
    "những": ["các", "loạt"],
    "các": ["những", "một số"],
    "cho hay": ["cho biết", "chia sẻ"],
    "cho biết": ["cho hay", "thông tin"],
    "giúp": ["hỗ trợ", "góp phần"],
    "người dân": ["mọi người", "công dân"],
    "trẻ em": ["trẻ nhỏ", "con trẻ"],
    "bác sĩ": ["chuyên gia", "thầy thuốc"],
    "nguyên nhân": ["lý do", "tác nhân"],
    "cần": ["nên", "phải"],
    "thường": ["hay", "thường xuyên"],
    "tuy nhiên": ["nhưng", "dẫu vậy"],
    "vì vậy": ["do đó", "bởi thế"],
    "có thể": ["khả năng", "dễ"],
    "phát hiện": ["nhận biết", "tìm ra"],
    "sức khỏe": ["thể trạng", "cơ thể"]
}

# Các từ cấm thay đổi (Thuật ngữ y tế nhạy cảm)
STOP_WORDS = ["tăng", "giảm", "không", "có", "đái tháo đường", "huyết áp", "ung thư", "tử vong"]

print("Bộ từ điển an toàn đã sẵn sàng.")

Bộ từ điển an toàn đã sẵn sàng.


In [11]:
def safe_augment_v6(text, numbers):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    augmented_sentences = []

    for sent in sentences:
        if not sent.strip(): continue

        words = word_tokenize(sent)
        changed = False
        new_words = []

        # Bước 1: Thử thay thế bằng từ điển đồng nghĩa
        for w in words:
            clean_w = w.lower().replace('_', ' ')
            if clean_w in SAFE_SYNONYMS and clean_w not in STOP_WORDS:
                replacement = random.choice(SAFE_SYNONYMS[clean_w])
                new_words.append(replacement)
                changed = True
            else:
                new_words.append(w)

        # Bước 2: Nếu chưa có thay đổi, thực hiện tiểu phẫu (thêm/bớt hư từ)
        if not changed and len(new_words) > 5:
            # Thử thêm từ "thật sự" hoặc "rất" vào trước tính từ hoặc đổi vị trí dấu phẩy
            idx = random.randint(1, len(new_words) - 2)
            if new_words[idx] not in STOP_WORDS and new_words[idx] not in numbers:
                new_words.insert(idx, "vốn")
                changed = True

        # Ghép lại câu
        aug_sent = " ".join(new_words).replace('_', ' ')
        augmented_sentences.append(aug_sent)

    # Hậu xử lý định dạng
    final_text = " ".join(augmented_sentences)
    final_text = re.sub(r'\s+([.,!?;])', r'\1', final_text)
    return final_text.strip()

In [12]:
base_dir = "/content/drive/MyDrive/MyNLPData"
input_path = os.path.join(base_dir, "train/dataset.jsonl")
output_path = os.path.join(base_dir, "test_v6_safe_10.jsonl")

if os.path.exists(input_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        original_data = [json.loads(next(f)) for _ in range(10)]

    augmented_results = []
    print("Bắt đầu xử lý 10 mẫu (V6 - Cam kết thay đổi 100%)...")

    for i, item in enumerate(original_data):
        new_source = safe_augment_v6(item['source'], item['src_numbers'])

        # Chống trùng lặp tuyệt đối (nếu đen đủi vẫn giống hệt thì sửa thủ công)
        if new_source == item['source']:
            new_source = "Theo ghi nhận, " + new_source

        new_item = item.copy()
        new_item['id'] = f"{item['id']}_v6_aug"
        new_item['source'] = new_source
        augmented_results.append(new_item)
        print(f"Xong mẫu {i+1}")

    with open(output_path, 'w', encoding='utf-8') as f_out:
        for entry in augmented_results:
            f_out.write(json.dumps(entry, ensure_ascii=False) + '\n')

    print(f"\nĐã lưu tại: {output_path}")
    print("\n--- KIỂM TRA MẪU SỐ 1 ---")
    print(f"GỐC: {original_data[0]['source'][:150]}...")
    print(f"MỚI: {augmented_results[0]['source'][:150]}...")

Bắt đầu xử lý 10 mẫu (V6 - Cam kết thay đổi 100%)...
Xong mẫu 1
Xong mẫu 2
Xong mẫu 3
Xong mẫu 4
Xong mẫu 5
Xong mẫu 6
Xong mẫu 7
Xong mẫu 8
Xong mẫu 9
Xong mẫu 10

Đã lưu tại: /content/drive/MyDrive/MyNLPData/test_v6_safe_10.jsonl

--- KIỂM TRA MẪU SỐ 1 ---
GỐC: Bỏ bữa sáng, ăn không đúng giờ, vừa ăn vừa xem tivi hay tiêu thụ nhiều thực phẩm có đường là những thói quen ăn uống gây hại cho sức khỏe trẻ.
Bỏ bữa ...
MỚI: Bỏ bữa sáng, ăn không đúng giờ, vừa ăn vừa xem tivi hay tiêu thụ nhiều thực phẩm có đường là các thói quen ăn uống gây hại cho sức khỏe trẻ. Bỏ bữa sá...


PHÂN TÍCH - SO SÁNH

In [15]:
import json
import os

# 1. Đường dẫn file
base_dir = "/content/drive/MyDrive/MyNLPData"
input_orig = os.path.join(base_dir, "train/dataset.jsonl")
input_aug = os.path.join(base_dir, "test_v6_safe_10.jsonl")
output_txt = os.path.join(base_dir, "compare.txt") # File kết quả so sánh

# 2. Hàm trích xuất source và ghi file
def compare_sources(orig_path, aug_path, txt_path, num_samples=10):
    if not os.path.exists(orig_path) or not os.path.exists(aug_path):
        print("Lỗi: Không tìm thấy file đầu vào.")
        return

    with open(orig_path, 'r', encoding='utf-8') as f_orig, \
         open(aug_path, 'r', encoding='utf-8') as f_aug, \
         open(txt_path, 'w', encoding='utf-8') as f_out: # Mở file txt để ghi

        orig_data = [json.loads(line) for line in f_orig]
        aug_data = [json.loads(line) for line in f_aug]

        for i in range(min(num_samples, len(aug_data))):
            # Tạo chuỗi nội dung cho từng mẫu
            header = f"--- MẪU SỐ {i+1} ---"
            sample_id = f"ID: {aug_data[i]['id']}"
            source_orig = f"\n[SOURCE GỐC]:\n{orig_data[i]['source']}"
            source_new = f"\n[SOURCE MỚI]:\n{aug_data[i]['source']}"
            footer = "-" * 50 + "\n\n"

            # 1. In ra màn hình để xem nhanh
            print(header)
            print(sample_id)
            print(source_orig)
            print(source_new)
            print(footer)

            # 2. Ghi vào file txt
            f_out.write(header + "\n")
            f_out.write(sample_id + "\n")
            f_out.write(source_orig + "\n")
            f_out.write(source_new + "\n")
            f_out.write(footer)

    print(f"Hoàn thành! Kết quả so sánh đã được lưu tại: {txt_path}")

# 3. Thực hiện trích xuất
compare_sources(input_orig, input_aug, output_txt)

--- MẪU SỐ 1 ---
ID: 5f920a1aeda2_v6_aug

[SOURCE GỐC]:
Bỏ bữa sáng, ăn không đúng giờ, vừa ăn vừa xem tivi hay tiêu thụ nhiều thực phẩm có đường là những thói quen ăn uống gây hại cho sức khỏe trẻ.
Bỏ bữa sáng là một trong những nguyên nhân hàng đầu gây thiếu tập trung, hiệu suất học tập giảm, ăn vặt không lành mạnh. Về lâu dài, thói quen này gây thiếu hụt năng lượng, chất dinh dưỡng như sắt, kẽm, vitamin dẫn đến mệt mỏi, giảm trí nhớ, ảnh hưởng phát triển thể chất (chiều cao, cân nặng), tăng nguy cơ béo phì do ăn bù, đau dạ dày.
Cha mẹ nên cho trẻ ăn bữa sáng đủ các nhóm chất đạm, chất béo, chất bột đường, vitamin và khoáng chất. Sự cân bằng này cung cấp năng lượng, xây dựng hệ miễn dịch, hấp thụ dưỡng chất cần thiết, phát triển sức khỏe toàn diện.
Ưu tiên món nóng, dễ tiêu hóa như cháo, phở, bánh mì kẹp, yến mạch để cung cấp năng lượng ổn định cho buổi sáng. Nên cho trẻ ăn sáng sau khi thức dậy khoảng 30 phút, tránh để bé vừa đi vừa ăn trên đường đến trường, gây khó chịu cho hệ tiêu

PHÂN TÍCH _ SO SÁNH - CHI TIẾT HƠN

In [16]:
import json
import os
import re
from underthesea import word_tokenize

# 1. Cấu hình đường dẫn
base_dir = "/content/drive/MyDrive/MyNLPData"
input_orig = os.path.join(base_dir, "train/dataset.jsonl")
input_aug = os.path.join(base_dir, "test_v6_safe_10.jsonl")
output_report = os.path.join(base_dir, "detailed_compare.txt")

def get_word_diff(orig_text, new_text):
    # Tách từ và chuẩn hóa về chữ thường để so sánh chính xác
    orig_words = set([w.lower().replace('_', ' ') for w in word_tokenize(orig_text)])
    new_words = set([w.lower().replace('_', ' ') for w in word_tokenize(new_text)])

    added = new_words - orig_words
    removed = orig_words - new_words

    return added, removed

def generate_detailed_report(orig_path, aug_path, report_path, num_samples=10):
    if not os.path.exists(orig_path) or not os.path.exists(aug_path):
        print("Lỗi: Không tìm thấy file đầu vào.")
        return

    with open(orig_path, 'r', encoding='utf-8') as f_orig, \
         open(aug_path, 'r', encoding='utf-8') as f_aug, \
         open(report_path, 'w', encoding='utf-8') as f_out:

        orig_data = [json.loads(line) for line in f_orig]
        aug_data = [json.loads(line) for line in f_aug]

        f_out.write("BÁO CÁO CHI TIẾT BIẾN ĐỔI DỮ LIỆU (DATA AUGMENTATION REPORT)\n")
        f_out.write("="*70 + "\n\n")

        for i in range(min(num_samples, len(aug_data))):
            s_orig = orig_data[i]['source']
            s_new = aug_data[i]['source']

            # Tính toán số lượng từ (split theo khoảng trắng cơ bản)
            cnt_orig = len(s_orig.split())
            cnt_new = len(s_new.split())

            # Tìm từ khác biệt (dùng NLP tokenizer)
            added, removed = get_word_diff(s_orig, s_new)

            # Chuẩn bị nội dung ghi file
            header = f"--- MẪU SỐ {i+1} [ID: {aug_data[i]['id']}] ---"
            stats = f"Số từ: Gốc ({cnt_orig}) | Mới ({cnt_new}) | Chênh lệch: {cnt_new - cnt_orig}"

            added_str = ", ".join(list(added)) if added else "Không có"
            removed_str = ", ".join(list(removed)) if removed else "Không có"

            # In ra console
            print(header)
            print(stats)
            print(f"Từ mới xuất hiện: {added_str}")
            print(f"Từ bị thay thế/mất: {removed_str}")
            print("-" * 50)

            # Ghi vào file
            f_out.write(header + "\n")
            f_out.write(stats + "\n\n")
            f_out.write(f"[+] TỪ MỚI XUẤT HIỆN:\n{added_str}\n\n")
            f_out.write(f"[-] TỪ BỊ THAY THẾ/MẤT:\n{removed_str}\n\n")
            f_out.write("[NỘI DUNG MỚI]:\n" + s_new[:300] + "...\n") # In đoạn đầu để check format
            f_out.write("-" * 70 + "\n\n")

    print(f"\nBáo cáo chi tiết đã được xuất ra: {report_path}")

# Thực hiện chạy report
generate_detailed_report(input_orig, input_aug, output_report)

--- MẪU SỐ 1 [ID: 5f920a1aeda2_v6_aug] ---
Số từ: Gốc (507) | Mới (524) | Chênh lệch: 17
Từ mới xuất hiện: một số, lý do, hỗ trợ, vốn, khả năng, loạt, thể trạng
Từ bị thay thế/mất: có thể, nguyên nhân, giúp, sức khỏe
--------------------------------------------------
--- MẪU SỐ 2 [ID: ff613141adaa_v6_aug] ---
Số từ: Gốc (767) | Mới (791) | Chênh lệch: 24
Từ mới xuất hiện: ra, hỗ trợ, tìm, thầy thuốc, vốn, hay, tác nhân, loạt, thể trạng, công dân, nên
Từ bị thay thế/mất: người dân, phát hiện, bác sĩ, tính, cần, mạn, giúp, cho hay, nguyên nhân, thường, sức khỏe
--------------------------------------------------
--- MẪU SỐ 3 [ID: 61c5e86d9782_v6_aug] ---
Số từ: Gốc (621) | Mới (633) | Chênh lệch: 12
Từ mới xuất hiện: thầy thuốc ngọc, chuyên gia, lý do, thầy thuốc, vốn, chuyên gia ngọc
Từ bị thay thế/mất: bác sĩ, có thể, ngọc, giúp, cần, cho hay, nguyên nhân, thường
--------------------------------------------------
--- MẪU SỐ 4 [ID: 4538b07e4d59_v6_aug] ---
Số từ: Gốc (664) | Mới (682) | 

In [17]:
SAFE_SYNONYMS = {
    # --- NHÓM 1: HƯ TỪ, CHỈ SỐ LƯỢNG & ĐỊNH DANH (30 từ) ---
    "những": ["các", "loạt", "một số", "danh sách"],
    "các": ["những", "loạt", "một vài", "các loại"],
    "mọi": ["tất cả", "mỗi", "toàn bộ", "hết thảy"],
    "nhiều": ["đông đảo", "hàng loạt", "không ít", "vô số"],
    "một số": ["vài", "một vài", "một ít", "danh sách"],
    "đa số": ["phần lớn", "hầu hết", "số đông", "phần đông"],
    "ít": ["hiếm", "không nhiều", "thiếu hụt", "hạn chế"],
    "mỗi": ["từng", "mọi", "mỗi một", "tất thảy"],
    "từng": ["mỗi", "từng bước", "mọi"],
    "loạt": ["những", "các", "chuỗi"],
    "vài": ["một vài", "một số", "ít"],
    "hầu hết": ["đa phần", "phần lớn", "gần như toàn bộ"],
    "phần lớn": ["đa số", "hầu hết", "số đông"],
    "toàn bộ": ["tất cả", "mọi", "tổng thể"],
    "người": ["cá nhân", "đối tượng", "chủ thể"],
    "việc": ["vấn đề", "công tác", "nhiệm vụ"],
    "sự": ["quá trình", "tình trạng", "diễn biến"],
    "cái": ["vật", "thứ", "món"],
    "thứ": ["loại", "món", "vật phẩm"],
    "loại": ["thứ", "dòng", "nhóm"],
    "nhóm": ["tập hợp", "đội ngũ", "phân khúc"],
    "đội ngũ": ["nhóm", "tập thể", "lực lượng"],
    "lực lượng": ["đội ngũ", "nhóm", "nhân sự"],
    "cả": ["tất cả", "cùng", "toàn bộ"],
    "chỉ": ["duy nhất", "độc nhất", "vẻn vẹn"],
    "luôn": ["thường xuyên", "liên tục", "mãi"],
    "vẫn": ["đang", "tiếp tục", "vốn"],
    "cũng": ["cùng", "đồng thời", "tương tự"],
    "đều": ["tất cả", "cùng", "hàng loạt"],
    "hầu như": ["gần như", "cơ bản", "xấp xỉ"],

    # --- NHÓM 2: ĐỘNG TỪ TƯỜNG THUẬT & THÔNG TIN (25 từ) ---
    "cho biết": ["cho hay", "chia sẻ", "thông tin", "khẳng định"],
    "cho hay": ["cho biết", "tiết lộ", "chia sẻ", "thông báo"],
    "nhận định": ["đánh giá", "cho rằng", "nêu quan điểm", "phân tích"],
    "chia sẻ": ["cho biết", "bày tỏ", "thông tin", "tiết lộ"],
    "khuyến cáo": ["khuyên", "đề nghị", "cảnh báo", "tư vấn"],
    "khuyên": ["khuyến nghị", "tư vấn", "hướng dẫn", "đề nghị"],
    "cảnh báo": ["lưu ý", "nhắc nhở", "khuyến cáo", "đưa tin"],
    "giải thích": ["lý giải", "phân tích", "trình bày", "minh chứng"],
    "nói": ["phát biểu", "chia sẻ", "trình bày", "khẳng định"],
    "khẳng định": ["xác nhận", "nhấn mạnh", "quả quyết"],
    "nhấn mạnh": ["khẳng định", "tập trung", "lưu ý"],
    "thông tin": ["thông báo", "cho biết", "cung cấp dữ liệu"],
    "cho rằng": ["đánh giá", "nhận định", "nghĩ là"],
    "đánh giá": ["nhận xét", "nhận định", "xem xét"],
    "phát biểu": ["nói", "chia sẻ", "trình bày"],
    "kêu gọi": ["vận động", "đề nghị", "thúc đẩy"],
    "thừa nhận": ["công nhận", "xác nhận", "đồng ý"],
    "phản ánh": ["nêu lên", "cho thấy", "mô tả"],
    "công bố": ["ra mắt", "thông báo", "trình làng"],
    "đề xuất": ["gợi ý", "đề nghị", "nêu ý tưởng"],
    "hướng dẫn": ["chỉ dẫn", "tư vấn", "giới thiệu"],
    "tư vấn": ["hướng dẫn", "khuyên", "gợi ý"],
    "đề nghị": ["yêu cầu", "đề xuất", "mong muốn"],
    "lưu ý": ["chú ý", "cẩn trọng", "ghi nhớ"],
    "xác nhận": ["khẳng định", "chứng thực", "đồng ý"],

    # --- NHÓM 3: TỪ NỐI & CHUYỂN Ý (20 từ) ---
    "tuy nhiên": ["nhưng", "dẫu vậy", "mặc dù vậy", "thế nhưng"],
    "vì vậy": ["do đó", "bởi thế", "cho nên", "hệ quả là"],
    "do đó": ["vì lẽ đó", "bởi vậy", "hệ quả là", "thành ra"],
    "ngoài ra": ["bên cạnh đó", "thêm vào đó", "chưa kể", "song song"],
    "đồng thời": ["song song đó", "cùng với đó", "kết hợp", "vừa là"],
    "mặt khác": ["hơn nữa", "vả lại", "thêm vào đó", "bên cạnh đó"],
    "đặc biệt": ["nhất là", "cụ thể là", "điển hình là", "trọng tâm là"],
    "hơn nữa": ["mặt khác", "vả lại", "thêm nữa"],
    "nhất là": ["đặc biệt", "cụ thể là", "điển hình"],
    "cụ thể": ["chi tiết", "rõ ràng là", "điển hình"],
    "ngược lại": ["trái lại", "mặt khác", "đối lập"],
    "thực tế": ["thực chất", "trên thực tế", "kết quả là"],
    "bởi vì": ["do", "vì", "tại"],
    "mặc dù": ["dẫu cho", "cho dù", "tuy rằng"],
    "thậm chí": ["ngay cả", "đến mức", "còn"],
    "ngoài": ["trừ", "bên cạnh", "không chỉ"],
    "thay vì": ["thay cho", "đổi lại", "thay thế"],
    "nhờ": ["dựa vào", "thông qua", "bằng"],
    "theo": ["dựa theo", "căn cứ", "chiếu theo"],
    "như": ["giống như", "tương tự", "điển hình"],

    # --- NHÓM 4: ĐỘNG TỪ & DANH TỪ CHUNG AN TOÀN (30 từ) ---
    "giúp": ["hỗ trợ", "góp phần", "tạo điều kiện", "giúp ích"],
    "hỗ trợ": ["giúp ích", "góp phần", "phụ trợ", "tương trợ"],
    "nguyên nhân": ["lý do", "tác nhân", "nguồn cơn", "gốc rễ"],
    "lý do": ["tác nhân", "nguyên cớ", "nguồn cơn", "lẽ"],
    "phát hiện": ["nhận biết", "tìm ra", "xác định", "nhận diện"],
    "nhận biết": ["nhận diện", "phát hiện", "tìm thấy", "xác định"],
    "sức khỏe": ["thể trạng", "cơ địa", "cơ thể", "thể chất"],
    "thể trạng": ["thể chất", "cơ địa", "sức khỏe", "tình trạng cơ thể"],
    "trẻ em": ["trẻ nhỏ", "con trẻ", "bé", "nhi đồng"],
    "bác sĩ": ["chuyên gia", "thầy thuốc", "y bác sĩ", "người điều trị"],
    "bệnh nhân": ["người bệnh", "người mắc bệnh", "ca bệnh", "đối tượng điều trị"],
    "điều trị": ["chữa trị", "xử lý", "can thiệp y tế", "hồi phục"],
    "kiểm tra": ["thăm khám", "xét nghiệm", "tầm soát", "rà soát"],
    "phương pháp": ["cách thức", "biện pháp", "phác đồ", "quy trình"],
    "nguy cơ": ["khả năng", "xác suất", "tỷ lệ", "rủi ro"],
    "ảnh hưởng": ["tác động", "hệ lụy", "biến chuyển", "chi phối"],
    "duy trì": ["giữ vững", "tiếp tục", "bảo toàn", "duy trì đều đặn"],
    "thực hiện": ["tiến hành", "triển khai", "áp dụng", "vận hành"],
    "sử dụng": ["dùng", "tiêu thụ", "vận dụng", "áp dụng"],
    "thường xuyên": ["hay", "luôn", "liên tục", "đều đặn"],
    "kịp thời": ["đúng lúc", "nhanh chóng", "sớm", "tức thì"],
    "hiệu quả": ["tác dụng", "kết quả tốt", "hữu ích", "khả quan"],
    "phát triển": ["mở rộng", "tiến triển", "nâng cao"],
    "thay đổi": ["biến đổi", "điều chỉnh", "cải tổ"],
    "điều chỉnh": ["thay đổi", "uốn nắn", "sửa đổi"],
    "ngăn chặn": ["hạn chế", "phòng ngừa", "đẩy lùi"],
    "phòng ngừa": ["ngăn chặn", "đề phòng", "tránh"],
    "kiểm soát": ["quản lý", "nắm giữ", "giám sát"],
    "cải thiện": ["nâng cao", "thúc đẩy", "làm tốt hơn"],
    "xử lý": ["giải quyết", "điều trị", "thực hiện"],

    # --- NHÓM 5: TÍNH TỪ & TRẠNG TỪ CHUNG (15 từ) ---
    "quan trọng": ["thiết yếu", "cốt lõi", "trọng yếu", "đáng chú ý"],
    "phổ biến": ["thường gặp", "đại trà", "quen thuộc", "rộng rãi"],
    "nhanh chóng": ["mau chóng", "tức thì", "ngay", "lẹ"],
    "ổn định": ["bình thường", "cân bằng", "duy trì", "bền vững"],
    "rõ rệt": ["nhìn thấy rõ", "đáng kể", "minh bạch", "hiển nhiên"],
    "đáng kể": ["rõ rệt", "to lớn", "nhiều", "vượt bậc"],
    "phức tạp": ["rắc rối", "khó khăn", "nhiều công đoạn", "gian truân"],
    "nghiêm trọng": ["nặng nề", "nguy cấp", "trầm trọng", "đáng báo động"],
    "nguy hiểm": ["rủi ro", "nguy kịch", "đáng ngại", "không an toàn"],
    "tự nhiên": ["nguyên bản", "vốn có", "thuần túy", "tự thân"],
    "hoàn toàn": ["tuyệt đối", "hết thảy", "trọn vẹn", "tất tần tật"],
    "an toàn": ["lành mạnh", "đảm bảo", "chắc chắn"],
    "khoa học": ["hợp lý", "chuẩn mực", "bài bản"],
    "đầy đủ": ["trọn vẹn", "hết thảy", "sung túc"],
    "cần thiết": ["thiết yếu", "quan trọng", "cấp bách"]
}

STOP_WORDS = [
    # Thuật ngữ bệnh & Trạng thái y tế
    "đái tháo đường", "tiểu đường", "huyết áp", "gan nhiễm mỡ", "xơ gan", "ung thư",
    "đột quỵ", "tim mạch", "vắc xin", "vaccine", "viêm", "nhiễm", "tử vong", "liệt",
    "zona", "hiv", "covid", "suy thận", "u não", "xuất huyết", "thai", "tử cung",

    # Hướng tác động (CỰC KỲ QUAN TRỌNG)
    "tăng", "giảm", "cao", "thấp", "nóng", "lạnh", "không", "có", "mặn", "ngọt",
    "lười", "năng", "đúng", "sai", "trái", "phải", "trước", "sau", "đến", "đi",

    # Đơn vị & Thời gian (Anchor words)
    "triệu", "vạn", "tỷ", "ngàn", "tuổi", "tháng", "năm", "ngày", "giờ", "phút", "giây",
    "mũi", "liều", "lần", "mmol", "mg", "g", "kg", "ml", "lít", "đơn vị", "%", "phần trăm",
    "sáng", "trưa", "chiều", "tối", "đêm",

    # Các thành phần chất
    "đường", "muối", "mỡ", "đạm", "béo", "protein", "vitamin", "sắt", "kẽm", "calo",
    "rau", "thịt", "cá", "trứng", "sữa", "nước"
]

In [18]:
import json
import os
import random
import re
from underthesea import word_tokenize

def final_safe_augment(text):
    # Tách đoạn văn thành các câu
    sentences = re.split(r'(?<=[.!?])\s+', text)
    augmented_sentences = []

    for sent in sentences:
        if not sent.strip(): continue

        words = word_tokenize(sent)
        new_words = []
        changed = False

        for w in words:
            clean_w = w.lower().replace('_', ' ')
            # Nếu từ nằm trong từ điển an toàn, thay thế nó
            if clean_w in SAFE_SYNONYMS:
                new_words.append(random.choice(SAFE_SYNONYMS[clean_w]))
                changed = True
            else:
                new_words.append(w)

        # Nếu sau khi duyệt hết mà câu vẫn chưa đổi, thực hiện chèn từ đệm
        if not changed and len(new_words) > 4:
            insert_pos = random.randint(1, len(new_words) - 1)
            # Chèn từ "vốn" hoặc "thực sự" vào vị trí ngẫu nhiên
            new_words.insert(insert_pos, random.choice(["vốn", "thực sự", "hiện"]))
            changed = True

        augmented_sentences.append(" ".join(new_words).replace('_', ' '))

    # Hậu xử lý định dạng dấu câu
    final_text = " ".join(augmented_sentences)
    final_text = re.sub(r'\s+([.,!?;])', r'\1', final_text)
    final_text = re.sub(r'\s+', ' ', final_text)

    return final_text.strip()

# Thực thi tăng cường 780 mẫu
base_dir = "/content/drive/MyDrive/MyNLPData"
input_path = os.path.join(base_dir, "train/dataset.jsonl")
output_path = os.path.join(base_dir, "augmented_780_safe_dictionary.jsonl")

if os.path.exists(input_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        data = [json.loads(line) for line in f]

    augmented_list = []
    print(f"Bắt đầu xử lý {len(data)} mẫu bằng từ điển an toàn mở rộng...")

    for item in data:
        new_source = final_safe_augment(item['source'])

        # Đảm bảo khác biệt 100% bằng cách kiểm tra chuỗi
        if new_source == item['source']:
            new_source = "Ghi nhận là, " + new_source

        new_item = item.copy()
        new_item['id'] = f"{item['id']}_safe_aug"
        new_item['source'] = new_source
        new_item['src_word_count'] = len(new_source.split())
        augmented_list.append(new_item)

    with open(output_path, 'w', encoding='utf-8') as f_out:
        for entry in augmented_list:
            f_out.write(json.dumps(entry, ensure_ascii=False) + '\n')

    print(f"Hoàn thành! Đã tạo 780 mẫu mới tại: {output_path}")

Bắt đầu xử lý 780 mẫu bằng từ điển an toàn mở rộng...
Hoàn thành! Đã tạo 780 mẫu mới tại: /content/drive/MyDrive/MyNLPData/augmented_780_safe_dictionary.jsonl


kiểm tra xem có thật sự thay đổi chưa

In [20]:
import json
import os
import re
from underthesea import word_tokenize
from difflib import SequenceMatcher

# 1. Cấu hình đường dẫn
base_dir = "/content/drive/MyDrive/MyNLPData"
input_orig = os.path.join(base_dir, "train/dataset.jsonl")
input_aug = os.path.join(base_dir, "augmented_780_safe_dictionary.jsonl") # File chứa 780 mẫu mới
output_report = os.path.join(base_dir, "final_audit_report.txt")

def calculate_similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def get_word_diff(orig_text, new_text):
    # Tách từ và chuẩn hóa để so sánh tập hợp
    orig_words = set([w.lower().replace('_', ' ') for w in word_tokenize(orig_text)])
    new_words = set([w.lower().replace('_', ' ') for w in word_tokenize(new_text)])

    added = new_words - orig_words
    removed = orig_words - new_words
    return added, removed

def generate_audit_report(orig_path, aug_path, report_path):
    if not os.path.exists(orig_path) or not os.path.exists(aug_path):
        print("Lỗi: Kiểm tra lại đường dẫn file .jsonl")
        return

    with open(orig_path, 'r', encoding='utf-8') as f_orig, \
         open(aug_path, 'r', encoding='utf-8') as f_aug, \
         open(report_path, 'w', encoding='utf-8') as f_out:

        orig_data = [json.loads(line) for line in f_orig]
        aug_data = [json.loads(line) for line in f_aug]

        f_out.write("BÁO CÁO KIỂM ĐỊNH CHẤT LƯỢNG TĂNG CƯỜNG DỮ LIỆU (780 SAMPLES)\n")
        f_out.write("Mục tiêu: Đảm bảo 100% mẫu có thay đổi và bảo toàn y tế\n")
        f_out.write("="*80 + "\n\n")

        fail_count = 0
        success_count = 0

        for i in range(len(aug_data)):
            s_orig = orig_data[i]['source']
            s_new = aug_data[i]['source']

            # 1. Kiểm tra mức độ giống nhau
            similarity = calculate_similarity(s_orig, s_new)
            is_identical = (s_orig.strip() == s_new.strip())

            # 2. Thống kê từ vựng
            cnt_orig = len(s_orig.split())
            cnt_new = len(s_new.split())
            added, removed = get_word_diff(s_orig, s_new)

            status = "[SUCCESS]" if not is_identical else "[FAIL - NO CHANGE]"
            if is_identical: fail_count += 1
            else: success_count += 1

            # 3. Ghi chi tiết từng mẫu
            f_out.write(f"MẪU SỐ {i+1} | {status} | ID: {aug_data[i]['id']}\n")
            f_out.write(f"- Độ tương đồng (Similarity): {similarity:.2%}\n")
            f_out.write(f"- Số từ: Gốc ({cnt_orig}) -> Mới ({cnt_new})\n")

            if not is_identical:
                f_out.write(f"- Từ mới: {', '.join(list(added)[:15])}...\n")
                f_out.write(f"- Từ mất: {', '.join(list(removed)[:15])}...\n")
            else:
                f_out.write("⚠️ CẢNH BÁO: Dữ liệu mẫu này giống hệt bản gốc!\n")

            f_out.write("-" * 40 + "\n")

            if (i + 1) % 100 == 0:
                print(f"Đã đối soát {i+1}/780 mẫu...")

        # 4. TỔNG KẾT CUỐI FILE
        summary = f"""
{'='*80}
TỔNG KẾT KIỂM ĐỊNH:
- Tổng số mẫu kiểm tra: {len(aug_data)}
- Số mẫu hợp lệ (có thay đổi): {success_count}
- Số mẫu lỗi (không thay đổi): {fail_count}
- Tỷ lệ thành công: {(success_count/len(aug_data)):.2%}
{'='*80}
        """
        f_out.write(summary)
        print(summary)

# Thực thi
generate_audit_report(input_orig, input_aug, output_report)

Đã đối soát 100/780 mẫu...
Đã đối soát 200/780 mẫu...
Đã đối soát 300/780 mẫu...
Đã đối soát 400/780 mẫu...
Đã đối soát 500/780 mẫu...
Đã đối soát 600/780 mẫu...
Đã đối soát 700/780 mẫu...

TỔNG KẾT KIỂM ĐỊNH:
- Tổng số mẫu kiểm tra: 780
- Số mẫu hợp lệ (có thay đổi): 780
- Số mẫu lỗi (không thay đổi): 0
- Tỷ lệ thành công: 100.00%
        


In [21]:
import json
import os
from collections import Counter
from underthesea import word_tokenize
import pandas as pd

# 1. Cấu hình đường dẫn
base_dir = "/content/drive/MyDrive/MyNLPData"
file_orig = os.path.join(base_dir, "train/dataset.jsonl")
file_aug = os.path.join(base_dir, "augmented_780_safe_dictionary.jsonl")

def analyze_augmentation_drift(path_orig, path_aug):
    change_tracker = Counter()
    original_word_counts = Counter()

    with open(path_orig, 'r', encoding='utf-8') as f_o, \
         open(path_aug, 'r', encoding='utf-8') as f_a:

        for line_o, line_a in zip(f_o, f_a):
            orig_text = json.loads(line_o)['source']
            aug_text = json.loads(line_a)['source']

            # Tách từ (giữ nguyên underscore để nhận diện từ ghép)
            words_o = word_tokenize(orig_text)
            words_a = word_tokenize(aug_text)

            # Chỉ so sánh khi độ dài danh sách từ tương đồng (tránh lệch index do chèn 'vốn')
            # Ở đây ta dùng set để tìm các từ bị mất đi và từ mới xuất hiện trong cùng 1 mẫu
            set_o = set([w.lower().replace('_', ' ') for w in words_o])
            set_a = set([w.lower().replace('_', ' ') for w in words_a])

            removed = set_o - set_a
            added = set_a - set_o

            for w in removed:
                change_tracker[w] += 1

    # Chuyển thành DataFrame để quan sát
    df = pd.DataFrame(change_tracker.most_common(20), columns=['Từ gốc bị thay đổi', 'Số lần xuất hiện'])

    print("\n" + "="*30)
    print("TOP 20 TỪ CÓ BIẾN ĐỘNG NHIỀU NHẤT")
    print("="*30)
    print(df.to_string(index=False))

    # Tính tổng lượng thay đổi
    total_changes = sum(change_tracker.values())
    print(f"\nTổng số lượt từ vựng bị thay thế trên 780 mẫu: {total_changes}")
    print(f"Trung bình mỗi mẫu có {total_changes/780:.2f} từ bị thay đổi ngữ nghĩa.")

# Thực thi phân tích
analyze_augmentation_drift(file_orig, file_aug)


TOP 20 TỪ CÓ BIẾN ĐỘNG NHIỀU NHẤT
Từ gốc bị thay đổi  Số lần xuất hiện
             nhiều               582
            bác sĩ               576
              giúp               520
           nguy cơ               512
              cũng               494
               chỉ               429
              việc               419
         kiểm soát               331
             người               329
                sự               327
           sử dụng               326
               vẫn               326
               mỗi               323
          đặc biệt               306
          hiệu quả               301
         tuy nhiên               281
             ngoài               273
          thay đổi               263
                cả               250
         phát hiện               243

Tổng số lượt từ vựng bị thay thế trên 780 mẫu: 19210
Trung bình mỗi mẫu có 24.63 từ bị thay đổi ngữ nghĩa.
